# V8 · Stage 1.4 — θ identifiability

**Acceptance criterion**: for each anchor, top-10% population span per θ
parameter (from `configs/deg_params/*.yaml`); flag `well_identified` when
span < 30% of search-bound width. Gate check: ≥ 3 of 5 θ parameters
well-identified per anchor.

**Expected outputs**:
- `outputs/results/theta_identifiability.md`
- `outputs/results/theta_identifiability.parquet`

## Method
The DE-fit step (`phase2_de_fit.py`) already computed identifiability from
the final DE population and saved it into each anchor's YAML under
`fitted_params.<param>.identifiability`. This notebook reads those and
produces a summary + verdict.


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

PROJ = Path("/home/hj/Desktop/PINNs")
DEG_DIR = PROJ / "configs" / "deg_params"

ANCHORS = ["CALB_0003", "CALB_0009", "CALB_0010", "CALB_0015",
           "EVE_0004", "REPT_0007", "REPT_0057"]

rows = []
for aid in ANCHORS:
    p = DEG_DIR / f"{aid}.yaml"
    if not p.exists():
        print(f"MISSING {p}"); continue
    y = yaml.safe_load(p.read_text())
    for name, info in y["fitted_params"].items():
        ident = info.get("identifiability", {})
        rows.append({
            "anchor": aid,
            "param": name,
            "value": info.get("value", np.nan),
            "encoding": info.get("encoding", ""),
            "span_of_full_range": ident.get("span_of_full_range", np.nan),
            "well_identified": bool(ident.get("well_identified", False)),
        })

df = pd.DataFrame(rows)
print(df.to_string())


       anchor           param         value encoding  span_of_full_range  well_identified
0   CALB_0003           k_SEI  2.116991e-11    log10            0.162408             True
1   CALB_0003           V_SEI  1.117338e-04   linear            0.406314            False
2   CALB_0003   D_SEI_solvent  9.955699e-21    log10            0.050162             True
3   CALB_0003       k_plating  2.175666e-12    log10            0.032125             True
4   CALB_0003  k_LAM_negative  3.062843e-10    log10            0.653655            False
5   CALB_0009           k_SEI  1.746585e-13    log10            0.520740            False
6   CALB_0009           V_SEI  1.202658e-04   linear            0.320371            False
7   CALB_0009   D_SEI_solvent  2.385937e-22    log10            0.080947             True
8   CALB_0009       k_plating  5.050943e-12    log10            0.021826             True
9   CALB_0009  k_LAM_negative  3.638846e-08    log10            0.651505            False
10  CALB_0

In [2]:
# Per-anchor: how many of 5 parameters are well-identified?
per_anchor = (df.groupby("anchor")["well_identified"].sum()
                .rename("n_well_identified"))
per_anchor = per_anchor.to_frame()
per_anchor["n_total"] = df.groupby("anchor")["param"].count()
per_anchor["pass_gate_ge_3_of_5"] = per_anchor["n_well_identified"] >= 3
per_anchor


,n_well_identified,n_total,pass_gate_ge_3_of_5
anchor,,,
CALB_0003,3,5,True
CALB_0009,2,5,False
CALB_0010,3,5,True
CALB_0015,5,5,True
EVE_0004,3,5,True
REPT_0007,2,5,False
REPT_0057,5,5,True


In [3]:
# Per-parameter: which parameters are ROBUSTLY identifiable across anchors?
per_param = (df.groupby("param")["well_identified"].mean().rename("frac_anchors_identified"))
per_param = per_param.to_frame()
per_param["mean_span_of_full_range"] = df.groupby("param")["span_of_full_range"].mean()
per_param["reliable_across_anchors"] = per_param["frac_anchors_identified"] >= 0.7
per_param.sort_values("mean_span_of_full_range")


,frac_anchors_identified,mean_span_of_full_range,reliable_across_anchors
param,,,
k_plating,0.857143,0.071861,True
D_SEI_solvent,1.000000,0.088447,True
V_SEI,0.571429,0.254721,False
k_SEI,0.428571,0.332533,False
k_LAM_negative,0.428571,0.440148,False


In [4]:
# Persist artifacts
df.to_parquet(PROJ / "outputs" / "results" / "theta_identifiability.parquet",
              index=False)

md_lines = ["# θ identifiability — from DE final populations",
            "",
            "## Per-anchor summary (out of 5 parameters)",
            "",
            "| Anchor | Well-identified | Gate (≥ 3 of 5)? |",
            "|---|---|---|"]
for aid, r in per_anchor.iterrows():
    md_lines.append(f"| {aid} | {int(r['n_well_identified'])} | "
                    f"{'PASS' if r['pass_gate_ge_3_of_5'] else 'FAIL'} |")
md_lines += ["",
             "## Per-parameter reliability (fraction of anchors where identified)",
             "",
             "| Parameter | Frac identified | Mean span | Reliable? |",
             "|---|---|---|---|"]
for name, r in per_param.iterrows():
    md_lines.append(f"| {name} | {r['frac_anchors_identified']:.2f} | "
                    f"{r['mean_span_of_full_range']:.3f} | "
                    f"{'YES' if r['reliable_across_anchors'] else 'NO'} |")

overall_pass = per_anchor["pass_gate_ge_3_of_5"].all()
md_lines += ["",
             f"## Overall gate: {'PASS' if overall_pass else 'FAIL'}",
             "",
             "If any anchor has < 3 well-identified params, the operator is "
             "conditioned on physically ambiguous inputs for that anchor's "
             "corpus. Report as a LIMITATION in the abstract's methodology "
             "paragraph; do not describe θ as a uniquely-identified physical "
             "parameter vector."]
report = "\n".join(md_lines)
(PROJ / "outputs" / "results" / "theta_identifiability.md").write_text(report)
print(report)


# θ identifiability — from DE final populations

## Per-anchor summary (out of 5 parameters)

| Anchor | Well-identified | Gate (≥ 3 of 5)? |
|---|---|---|
| CALB_0003 | 3 | PASS |
| CALB_0009 | 2 | FAIL |
| CALB_0010 | 3 | PASS |
| CALB_0015 | 5 | PASS |
| EVE_0004 | 3 | PASS |
| REPT_0007 | 2 | FAIL |
| REPT_0057 | 5 | PASS |

## Per-parameter reliability (fraction of anchors where identified)

| Parameter | Frac identified | Mean span | Reliable? |
|---|---|---|---|
| D_SEI_solvent | 1.00 | 0.088 | YES |
| V_SEI | 0.57 | 0.255 | NO |
| k_LAM_negative | 0.43 | 0.440 | NO |
| k_SEI | 0.43 | 0.333 | NO |
| k_plating | 0.86 | 0.072 | YES |

## Overall gate: FAIL

If any anchor has < 3 well-identified params, the operator is conditioned on physically ambiguous inputs for that anchor's corpus. Report as a LIMITATION in the abstract's methodology paragraph; do not describe θ as a uniquely-identified physical parameter vector.


## Verdict marker

Update per the overall gate above:

- [ ] **PASS** — all 7 anchors have ≥ 3 of 5 well-identified parameters
- [ ] **PASS WITH LIMITATIONS** — some anchors below gate; caveat added to abstract
- [ ] **FAIL** — most anchors below gate; refit or reformulate loss (Stage 4.1)
